<a href="https://colab.research.google.com/github/Husseinhhameed/Mobile-CoHAtNet/blob/main/Mobile_CoHAtnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transforms3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.4 MB/s eta 0:00:00


In [ ]:
!pip install einops

In [ ]:
!pip install kornia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.3/833.3 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transforms3d
!pip install einops
!pip install kornia

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.4/899.4 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.5 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
import pickle
import time
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision import transforms
from torchvision.transforms import Compose, RandomHorizontalFlip, RandomRotation, ToTensor
from torchvision.transforms.functional import to_tensor
from torchvision.utils import make_grid

from einops import rearrange
from einops.layers.torch import Rearrange
from sklearn.model_selection import KFold
from tqdm import tqdm
import transforms3d.quaternions as txq
import matplotlib.pyplot as plt
from kornia.geometry.conversions import quaternion_to_rotation_matrix

In [ ]:
def compute_ABC(w_t_c, c_R_w, w_t_chat, chat_R_w, c_n, eye):
    """
    Computes A, B, and C matrix given estimated and ground truth poses
    and normal vector n.
    `w_t_c` and `w_t_chat` must have shape (batch_size, 3, 1).
    `c_R_w` and `chat_R_w` must have shape (batch_size, 3, 3).
    `n` must have shape (3, 1).
    `eye` is the (3, 3) identity matrix on the proper device.
    """
    # Ensure all inputs are float32
    w_t_c       = w_t_c.float()
    c_R_w       = c_R_w.float()
    w_t_chat    = w_t_chat.float()
    chat_R_w    = chat_R_w.float()
    c_n         = c_n.float()
    eye         = eye.float()

    # chat_t_c: predicted camera translation relative to ground-truth camera
    chat_t_c = chat_R_w @ (w_t_c - w_t_chat)
    # chat_R_c: predicted camera rotation relative to ground-truth camera
    chat_R_c = chat_R_w @ c_R_w.transpose(1, 2)

    A = eye - chat_R_c
    C = c_n @ chat_t_c.transpose(1, 2)
    B = C @ A

    A = A @ A.transpose(1, 2)
    B = B + B.transpose(1, 2)
    C = C @ C.transpose(1, 2)

    return A, B, C


In [ ]:
class GlobalHomographyLoss(nn.Module):
    def __init__(self, xmin, xmax, device='cpu'):
        """
        `xmin` is the minimum distance of observations across all frames.
        `xmax` is the maximum distance of observations across all frames.
        """
        super().__init__()
        # convert to tensor on the given device
        xmin = torch.tensor(xmin, dtype=torch.float32, device=device)
        xmax = torch.tensor(xmax, dtype=torch.float32, device=device)

        # compute weighting factors from dataset min / max
        self.B_weight = torch.log(xmin / xmax) / (xmax - xmin)
        self.C_weight = xmin * xmax

        # normal vector of the plane
        self.c_n = torch.tensor([0, 0, -1], dtype=torch.float32, device=device).view(3, 1)
        self.eye = torch.eye(3, device=device)

    def forward(self, batch):
        # compute A, B, C for the entire batch
        A, B, C = compute_ABC(
            batch['w_t_c'],
            batch['c_R_w'],
            batch['w_t_chat'],
            batch['chat_R_w'],
            self.c_n,
            self.eye
        )
        # combine them
        error = A + B * self.B_weight + C / self.C_weight
        # sum diagonal and average across batch
        error = error.diagonal(dim1=1, dim2=2).sum(dim=1).mean()
        return error



In [ ]:
def convert_to_transformation_matrix(translation, quaternion):
    """
    Converts a translation vector and quaternion to a 4x4 transformation matrix.
    """
    R = quaternion_to_rotation_matrix(quaternion)
    T = torch.eye(4)
    T[:3, :3] = R
    T[:3, 3]  = translation
    return T

In [ ]:
def load_and_process_poses(data_dir, seqs, train=True, real=False, vo_lib='orbslam'):
    """
    Loads and processes pose data from the given directory.
    """
    ps       = {}   # per-sequence raw pose arrays
    vo_stats = {}
    all_poses = []

    for seq in seqs:
        seq_dir      = os.path.join(data_dir, seq)
        p_filenames  = [n for n in os.listdir(seq_dir) if n.find('pose') >= 0]

        if real:
            # Load real dataset poses
            pose_file = os.path.join(data_dir, f'{vo_lib}_poses', seq)
            pss       = np.loadtxt(pose_file)
            frame_idx = pss[:, 0].astype(int)
            if vo_lib == 'libviso2':
                frame_idx -= 1
            ps[seq] = pss[:, 1:13]
            vo_stats_filename = os.path.join(seq_dir, f'{vo_lib}_vo_stats.pkl')
            with open(vo_stats_filename, 'rb') as f:
                vo_stats[seq] = pickle.load(f)

        else:
            # Synthetic dataset
            frame_idx = np.array(range(len(p_filenames)), dtype=int)
            pss = [
                np.loadtxt(os.path.join(seq_dir, f'frame-{i:06d}.pose.txt')).flatten()[:12]
                for i in frame_idx
                if os.path.exists(os.path.join(seq_dir, f'frame-{i:06d}.pose.txt'))
            ]
            ps[seq]         = np.asarray(pss)
            vo_stats[seq]   = {'R': np.eye(3), 't': np.zeros(3), 's': 1}

        all_poses.append(ps[seq])

    # Stack all poses from all sequences
    all_poses = np.vstack(all_poses)
    pose_stats_filename = os.path.join(data_dir, 'pose_stats.txt')

    # compute or load translation stats
    if train and not real:
        mean_t = np.mean(all_poses[:, [3, 7, 11]], axis=0)
        std_t  = np.std(all_poses[:, [3, 7, 11]], axis=0)
        np.savetxt(pose_stats_filename, np.vstack((mean_t, std_t)), fmt='%8.7f')
    else:
        mean_t, std_t = np.loadtxt(pose_stats_filename)

    # process each sequence's poses
    processed_poses = []
    for seq in seqs:
        pss = process_poses(
            poses_in=ps[seq],
            mean_t=mean_t,
            std_t=std_t,
            align_R=vo_stats[seq]['R'],
            align_t=vo_stats[seq]['t'],
            align_s=vo_stats[seq]['s']
        )
        processed_poses.append(pss)

    return np.vstack(processed_poses)



In [ ]:
def process_poses(poses_in, mean_t, std_t, align_R, align_t, align_s):
    """
    Aligns and normalizes poses, returning Nx7 (tx,ty,tz,qx,qy,qz,qw).
    """
    poses_out = np.zeros((len(poses_in), 7))
    # translation is columns 3,7,11 from the 3x4 matrix
    poses_out[:, 0:3] = poses_in[:, [3, 7, 11]]

    for i in range(len(poses_out)):
        R = poses_in[i].reshape((3, 4))[:3, :3]
        # align rotation
        R_aligned   = np.dot(align_R, R)
        q          = txq.mat2quat(R_aligned)
        q          *= np.sign(q[0])  # ensure consistent hemisphere
        poses_out[i, 3:] = q

        # align & scale translation
        t = poses_out[i, :3] - align_t
        t = align_s * np.dot(align_R, t[:, np.newaxis]).squeeze()
        poses_out[i, :3] = t

    # normalize translation
    poses_out[:, :3] -= mean_t
    poses_out[:, :3] /= std_t
    return poses_out



In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [ ]:
class FireDataset(Dataset):
    def __init__(self, root_dir, xmin_percentile=0.025, xmax_percentile=0.975, transform=None):
        self.root_dir          = root_dir
        self.transform         = transform
        self.xmin_percentile   = xmin_percentile
        self.xmax_percentile   = xmax_percentile

        # Load sequence folders and all sample paths
        self.seqs    = [seq for seq in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, seq))]
        self.samples = self._load_samples()

        # Check for precomputed depth stats
        stats_file = os.path.join(root_dir, 'depth_stats.json')
        if os.path.exists(stats_file):
            with open(stats_file, 'r') as f:
                stats = json.load(f)
                self.global_xmin = stats['global_xmin']
                self.global_xmax = stats['global_xmax']
        else:
            # compute on a subset of images
            self.global_depths = self._compute_global_depths()
            self.global_xmin   = np.percentile(self.global_depths, self.xmin_percentile * 100)
            self.global_xmax   = np.percentile(self.global_depths, self.xmax_percentile * 100)
            with open(stats_file, 'w') as f:
                json.dump({'global_xmin': self.global_xmin, 'global_xmax': self.global_xmax}, f)
            del self.global_depths

        # load and process poses
        self.processed_poses = self._load_processed_poses()

        # ensure lengths match
        min_length = min(len(self.samples), self.processed_poses.shape[0])
        self.samples         = self.samples[:min_length]
        self.processed_poses = self.processed_poses[:min_length]

    def _load_samples(self):
        """
        Loads file paths for color, depth, pose, and imu per frame.
        """
        samples = []
        for seq_folder in self.seqs:
            seq_path    = os.path.join(self.root_dir, seq_folder)
            color_files = sorted([f for f in os.listdir(seq_path) if f.endswith('.color.png')])
            depth_files = sorted([f for f in os.listdir(seq_path) if f.endswith('.depth.png')])
            pose_files  = sorted([f for f in os.listdir(seq_path) if f.endswith('.pose.txt')])
            imu_files   = sorted([f for f in os.listdir(seq_path) if f.endswith('.imu.txt')])

            # make sure your folder has the same number of color, depth, pose, and imu files
            # also ensure they correspond to the same frame number
            for c, d, p, imu in zip(color_files, depth_files, pose_files, imu_files):
                samples.append((
                    os.path.join(seq_path, c),
                    os.path.join(seq_path, d),
                    os.path.join(seq_path, p),
                    os.path.join(seq_path, imu)
                ))
        return samples

    def _compute_global_depths(self):
        """
        Computes global depth statistics by loading a subset of depth images.
        """
        num_samples    = len(self.samples)
        sample_indices = random.sample(range(num_samples), min(100, num_samples))
        all_depths     = []

        for idx in tqdm(sample_indices, desc="Computing global depths"):
            _, depth_path, _, _ = self.samples[idx]
            depth_image = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
            valid_depths = depth_image[depth_image > 0]
            all_depths.extend(valid_depths.flatten())

        all_depths = np.array(all_depths)
        return all_depths

    def _load_processed_poses(self):
        """
        Loads and processes pose data for all sequences.
        """
        return load_and_process_poses(self.root_dir, self.seqs)

    def __getitem__(self, idx):
        if idx >= len(self.processed_poses):
            raise IndexError(f"Index {idx} out of bounds for processed poses of size {len(self.processed_poses)}")

        color_path, depth_path, _, imu_path = self.samples[idx]

        # Load color + depth
        color_image = cv2.imread(color_path, cv2.IMREAD_COLOR)
        depth_image = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)

        # load the Nx7 pose
        pose_matrix = self.processed_poses[idx]

        # ------- NEW: Load IMU data (6 values) -------
        imu_data = np.loadtxt(imu_path, dtype=np.float32)  # shape: [6]
        imu_data = torch.from_numpy(imu_data)              # convert to tensor

        # transforms for color/depth (if requested)
        if self.transform:
            color_image = self.transform(color_image)
            # normalize depth to [0..255], then to float in [0..1]
            if depth_image.max() > 0:
                depth_image = (depth_image / depth_image.max() * 255).astype(np.uint8)
            depth_image = self.transform(depth_image)

        # Return color_image, depth_image, pose_matrix, imu_data
        return color_image, depth_image, torch.from_numpy(pose_matrix).float(), imu_data

    def __len__(self):
        return len(self.samples)

In [ ]:
def depthwise_separable_conv(inp, oup, image_size, downsample=False):
    stride = 2 if downsample else 1
    return nn.Sequential(
        nn.Conv2d(inp, inp, 3, stride, 1, groups=inp, bias=False),  # Depthwise
        nn.BatchNorm2d(inp),
        nn.GELU(),
        nn.Conv2d(inp, oup, 1, 1, 0, bias=False),                    # Pointwise
        nn.BatchNorm2d(oup),
        nn.GELU(),
    )


class PreNorm(nn.Module):
    def __init__(self, dim, fn, norm):
        super().__init__()
        self.norm = norm(dim)
        self.fn   = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class SE(nn.Module):
    def __init__(self, inp, oup, expansion=0.25):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(oup, int(inp*expansion), bias=False),
            nn.GELU(),
            nn.Linear(int(inp*expansion), oup, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class MBConv(nn.Module):
    def __init__(self, inp, oup, image_size, downsample=False, expansion=4):
        super().__init__()
        self.downsample = downsample
        stride = 2 if downsample else 1
        hidden_dim = int(inp * expansion)

        if self.downsample:
            self.pool = nn.MaxPool2d(3, 2, 1)
            self.proj = nn.Conv2d(inp, oup, 1, 1, 0, bias=False)

        if expansion == 1:
            self.conv = nn.Sequential(
                nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.GELU(),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            )
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(inp, hidden_dim, 1, stride, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.GELU(),
                nn.Conv2d(hidden_dim, hidden_dim, 3, 1, 1, groups=hidden_dim, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.GELU(),
                SE(inp, hidden_dim),
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
            )

        self.conv = PreNorm(inp, self.conv, nn.BatchNorm2d)

    def forward(self, x):
        if self.downsample:
            return self.proj(self.pool(x)) + self.conv(x)
        else:
            return x + self.conv(x)


class Attention(nn.Module):
    def __init__(self, inp, oup, image_size, heads=4, dim_head=32, dropout=0.):
        super().__init__()
        inner_dim  = dim_head * heads
        project_out = not (heads == 1 and dim_head == inp)

        self.ih, self.iw = image_size
        self.heads = heads
        self.scale = dim_head**-0.5

        # relative position bias
        self.relative_bias_table = nn.Parameter(torch.zeros((2*self.ih-1)*(2*self.iw-1), heads))
        coords = torch.meshgrid((torch.arange(self.ih), torch.arange(self.iw)))
        coords = torch.flatten(torch.stack(coords), 1)
        relative_coords = coords[:, :, None] - coords[:, None, :]
        relative_coords[0] += self.ih - 1
        relative_coords[1] += self.iw - 1
        relative_coords[0] *= (2*self.iw - 1)
        relative_coords = rearrange(relative_coords, 'c h w -> h w c')
        relative_index  = relative_coords.sum(-1).flatten().unsqueeze(1)
        self.register_buffer("relative_index", relative_index)

        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(inp, inner_dim*3, bias=False)

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, oup),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        relative_bias = self.relative_bias_table.gather(
            0, self.relative_index.repeat(1, self.heads)
        )
        relative_bias = rearrange(relative_bias, '(h w) c -> 1 c h w',
                                  h=self.ih*self.iw, w=self.ih*self.iw)
        dots = dots + relative_bias

        attn = self.attend(dots)
        out  = torch.matmul(attn, v)
        out  = rearrange(out, 'b h n d -> b n (h d)')
        out  = self.to_out(out)
        return out


class HAttention(nn.Module):
    def __init__(self, inp, oup, image_size, heads=4, dim_head=32, dropout=0.):
        super().__init__()
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == inp)

        self.ih, self.iw = image_size
        self.heads       = heads
        self.scale       = dim_head**-0.5

        self.relative_bias_table = nn.Parameter(torch.zeros((2*self.ih-1)*(2*self.iw-1), heads))
        coords = torch.meshgrid((torch.arange(self.ih), torch.arange(self.iw)))
        coords = torch.flatten(torch.stack(coords), 1)
        relative_coords = coords[:, :, None] - coords[:, None, :]
        relative_coords[0] += self.ih - 1
        relative_coords[1] += self.iw - 1
        relative_coords[0] *= (2*self.iw - 1)
        relative_coords = rearrange(relative_coords, 'c h w -> h w c')
        relative_index  = relative_coords.sum(-1).flatten().unsqueeze(1)
        self.register_buffer("relative_index", relative_index)

        self.attend = nn.Softmax(dim=-1)
        self.to_qk  = nn.Linear(inp, inner_dim*2, bias=False)

        self.to_out = nn.Sequential(
            nn.Linear(oup, oup),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x, mbconv):
        qk = self.to_qk(x).chunk(2, dim=-1)
        q, k = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qk)
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        relative_bias = self.relative_bias_table.gather(
            0, self.relative_index.repeat(1, self.heads)
        )
        relative_bias = rearrange(relative_bias, '(h w) c -> 1 c h w',
                                  h=self.ih*self.iw, w=self.ih*self.iw)
        dots = dots + relative_bias

        attn = self.attend(dots)
        v = rearrange(mbconv, 'b c ih iw -> b (ih iw) c')
        v = rearrange(v, 'b n (h c) -> b h n c', h=self.heads)

        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.to_out(out)
        return out


class Transformer(nn.Module):
    def __init__(self, inp, oup, image_size, heads=4, dim_head=32, downsample=False, dropout=0.):
        super().__init__()
        hidden_dim = int(inp*4)
        self.ih, self.iw = image_size
        self.downsample  = downsample

        if self.downsample:
            self.pool = nn.MaxPool2d(3, 2, 1)
            self.proj = nn.Conv2d(inp, oup, 1, 1, 0, bias=False)

        self.attn = Attention(inp, oup, image_size, heads, dim_head, dropout)
        self.ff   = FeedForward(oup, hidden_dim, dropout)

        self.attn = nn.Sequential(
            Rearrange('b c ih iw -> b (ih iw) c'),
            PreNorm(inp, self.attn, nn.LayerNorm),
            Rearrange('b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
        )

        self.ff = nn.Sequential(
            Rearrange('b c ih iw -> b (ih iw) c'),
            PreNorm(oup, self.ff, nn.LayerNorm),
            Rearrange('b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
        )

    def forward(self, x):
        if self.downsample:
            x = self.proj(self.pool(x)) + self.attn(self.pool(x))
        else:
            x = x + self.attn(x)
        x = x + self.ff(x)
        return x


class HTransformer(nn.Module):
    def __init__(self, inp, oup, image_size, heads=4, dim_head=32, downsample=False, dropout=0.):
        super().__init__()
        hidden_dim    = int(inp*4)
        self.ih, self.iw = image_size
        self.downsample  = downsample
        self.layer_norm  = nn.LayerNorm(inp)
        self.MBConv      = MBConv(inp, oup, image_size, downsample)

        if self.downsample:
            self.pool1 = nn.MaxPool2d(3, 2, 1)
            self.pool2 = nn.MaxPool2d(3, 2, 1)
            self.proj  = nn.Conv2d(inp, oup, 1, 1, 0, bias=False)

        self.attn = HAttention(inp, oup, image_size, heads, dim_head, dropout)
        self.ff   = FeedForward(oup, hidden_dim, dropout)

        self.ff   = nn.Sequential(
            Rearrange('b c ih iw -> b (ih iw) c'),
            PreNorm(oup, self.ff, nn.LayerNorm),
            Rearrange('b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
        )

    def forward(self, x):
        mbconv = self.MBConv(x)
        if self.downsample:
            pool1  = self.pool2(x)
            pool1  = rearrange(pool1, 'b c ih iw -> b (ih iw) c')
            norm1  = self.layer_norm(pool1)
            attn1  = self.attn(norm1, mbconv)
            attn1  = rearrange(attn1, 'b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
            out1   = self.proj(self.pool1(x)) + attn1
        else:
            xx     = rearrange(x, 'b c ih iw -> b (ih iw) c')
            norm1  = self.layer_norm(xx)
            attn1  = self.attn(norm1, mbconv)
            attn1  = rearrange(attn1, 'b (ih iw) c -> b c ih iw', ih=self.ih, iw=self.iw)
            out1   = x + attn1
        x = out1 + self.ff(out1)
        return x


In [ ]:
class CoHAtNet(nn.Module):
    def __init__(self, image_size, in_channels, num_blocks, channels,
                 num_classes=1000, block_types=['C','C','H','H']):
        super().__init__()
        ih, iw = image_size
        block = {'C': MBConv, 'T': Transformer, 'H': HTransformer}

        # Stages
        self.s0 = self._make_layer(depthwise_separable_conv, in_channels, channels[0], num_blocks[0], (ih//2, iw//2))
        self.s1 = self._make_layer(block[block_types[0]], channels[0], channels[1], num_blocks[1], (ih//4, iw//4))
        self.s2 = self._make_layer(block[block_types[1]], channels[1], channels[2], num_blocks[2], (ih//8, iw//8))
        self.s3 = self._make_layer(block[block_types[2]], channels[2], channels[3], num_blocks[3], (ih//16, iw//16))
        self.s4 = self._make_layer(block[block_types[3]], channels[3], channels[4], num_blocks[4], (ih//32, iw//32))

        self.pool = nn.AvgPool2d(ih//32, 1)

        # -------- NEW: IMU Embedding MLP --------
        self.imu_fc = nn.Sequential(
            nn.Linear(6, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU()
        )

        # After the pool, we get [B, channels[-1]].
        # The code does cat(x, x), so dimension doubles => 2*channels[-1].
        # Then we add +128 for IMU embedding => total_in
        total_in  = (channels[-1]*2) + 128
        hidden_dim = 1024

        self.fc1  = nn.Linear(total_in, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(hidden_dim, hidden_dim//2)
        self.fc3  = nn.Linear(hidden_dim//2, 7)  # pose: x, y, z, qx, qy, qz, qw

    def forward(self, x, imu=None):
        # 1) Forward pass through each stage
        x = self.s0(x)
        x = self.s1(x)
        x = self.s2(x)
        x = self.s3(x)
        x = self.s4(x)

        # 2) Global pooling => shape [B, channels[-1]]
        x = self.pool(x).view(-1, x.shape[1])

        # 3) Original code: we cat(x, x), doubling channels
        x = torch.cat((x, x), dim=1)  # shape [B, 2*channels[-1]]

        # 4) If we have IMU data, embed it and cat
        if imu is not None:
            imu_embed = self.imu_fc(imu)            # shape [B, 128]
            x = torch.cat((x, imu_embed), dim=1)    # shape [B, 2*channels[-1] + 128]

        # 5) Final MLP layers
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

    def _make_layer(self, block, inp, oup, depth, image_size):
        layers = nn.ModuleList([])
        for i in range(depth):
            if i == 0:
                layers.append(block(inp, oup, image_size, downsample=True))
            else:
                layers.append(block(oup, oup, image_size))
        return nn.Sequential(*layers)


def cohatnet_mobile():
    num_blocks = [1, 1, 2, 2, 1]            # Reduced depth
    channels   = [32, 48, 96, 192, 384]     # Reduced width
    return CoHAtNet(
        image_size=(256, 256),
        in_channels=3,
        num_blocks=num_blocks,
        channels=channels,
        num_classes=7
    )


In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
def batch_to_device(batch, device):
    """
    If `device` is not 'cpu', moves all data in batch to the GPU.
    """
    if device != 'cpu':
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                batch[key] = value.to(device)
            elif isinstance(value[0], torch.Tensor):
                for index_value, value_value in enumerate(value):
                    value[index_value] = value_value.to(device)
    return batch

In [ ]:
if __name__ == '__main__':
    # Adjust path as needed
    root_dir = '/content/drive/MyDrive/7-Scense/office'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')

    dataset = FireDataset(root_dir=root_dir, transform=transform)
    global_loss_fn = GlobalHomographyLoss(xmin=dataset.global_xmin, xmax=dataset.global_xmax, device=device)

    kfold     = KFold(n_splits=5, shuffle=True, random_state=42)
    num_epochs = 200
    best_val_loss = float('inf')

    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        print(f'Fold {fold + 1}')

        train_sampler = SubsetRandomSampler(train_idx)
        val_sampler   = SubsetRandomSampler(val_idx)

        train_loader = DataLoader(dataset, batch_size=8, sampler=train_sampler,
                                  num_workers=0, pin_memory=False)
        val_loader   = DataLoader(dataset, batch_size=8, sampler=val_sampler,
                                  num_workers=0, pin_memory=False)

        model     = cohatnet_mobile().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 'min', patience=4, factor=0.1
        )

        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0

            for color_images, depth_images, poses, imu_data in tqdm(train_loader):
                color_images = color_images.to(device)
                poses        = poses.to(device)   # shape [B, 7]
                imu_data     = imu_data.to(device)  # shape [B, 6]

                optimizer.zero_grad()

                # Pass IMU data to the model
                outputs = model(color_images, imu=imu_data)

                # Separate translation and quaternion
                pred_translation = outputs[:, :3]
                pred_quaternion = outputs[:, 3:]
                pred_quaternion = F.normalize(pred_quaternion, p=2, dim=-1)

                # Prepare batch for loss
                batch = {
                    'w_t_c':     pred_translation.unsqueeze(-1),
                    'c_R_w':     quaternion_to_rotation_matrix(pred_quaternion),
                    'w_t_chat':  poses[:, :3].unsqueeze(-1),
                    'chat_R_w':  quaternion_to_rotation_matrix(poses[:, 3:])
                }

                loss = global_loss_fn(batch)
                loss.backward()
                optimizer.step()

                train_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)

            # Validation
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for color_images, depth_images, poses, imu_data in val_loader:
                    color_images = color_images.to(device)
                    poses        = poses.to(device)
                    imu_data     = imu_data.to(device)

                    outputs = model(color_images, imu=imu_data)
                    pred_translation = outputs[:, :3]
                    pred_quaternion = outputs[:, 3:]
                    pred_quaternion = F.normalize(pred_quaternion, p=2, dim=-1)

                    batch = {
                        'w_t_c':     pred_translation.unsqueeze(-1),
                        'c_R_w':     quaternion_to_rotation_matrix(pred_quaternion),
                        'w_t_chat':  poses[:, :3].unsqueeze(-1),
                        'chat_R_w':  quaternion_to_rotation_matrix(poses[:, 3:])
                    }

                    loss = global_loss_fn(batch)
                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_loader)
            scheduler.step(avg_val_loss)

            print(f'Epoch {epoch + 1}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

            # Save the best model
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), 'best_cohatnet_model.pth')
                print(f'Best model saved at epoch {epoch + 1} with validation loss: {avg_val_loss:.4f}')

            torch.cuda.empty_cache()

Using device: cuda


/usr/local/lib/python3.10/dist-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Fold 1


100%|██████████| 1000/1000 [09:15<00:00,  1.80it/s]


Epoch 1, Train Loss: 0.5094, Val Loss: 0.1530
Best model saved at epoch 1 with validation loss: 0.1530


...

Epoch 99, Train Loss: 0.0001, Val Loss: 0.0005


100%|██████████| 1000/1000 [04:55<00:00,  3.39it/s]


Epoch 100, Train Loss: 0.0001, Val Loss: 0.0005


 75%|███████▌  | 754/1000 [03:41<01:12,  3.40it/s]


KeyboardInterrupt: 

In [ ]:
import torch

# Load your trained model
model = cohatnet_mobile()
model.load_state_dict(torch.load('/content/drive/MyDrive/best_cohatnet_model.pth'))
model.eval()

# Define an example input with the same dimensions as used during training
example_input = torch.randn(1, 3, 256, 256)

# Trace the model
scripted_model = torch.jit.trace(model, example_input)


<ipython-input-14-67107c906781>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/content/drive/MyDrive/best_cohatnet_model.pth'))


In [ ]:
# Define the transformation for the images
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [ ]:
torch.jit.save(scripted_model, 'optimized_cohatnet.pt')

# Download the file for deployment
from google.colab import files
files.download('optimized_cohatnet.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>